STAGE 5.1
# HAND ROI EXPERIMENTAL PROTOCOL & DATASET MANIFEST

In [3]:
# =============================================================================
# CISLR — STAGE 5.1
# HAND ROI EXPERIMENTAL PROTOCOL & DATASET MANIFEST
# =============================================================================
#
# PURPOSE:
#   Freeze the experimental conditions for Stage 5:
#
#       FULL FRAME  vs  HAND ROI
#
#   This stage DOES NOT:
#       - detect hands
#       - extract ROIs
#       - train a model
#       - perform retrieval
#       - convert to OpenVINO
#
#   It only establishes the reproducible Stage 5 protocol and manifests.
#
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import platform
import sys
import random
import time
from datetime import datetime

# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path("/home/dipshikha/Music/cao/CISLR_RESEARCH")

DATASET_ROOT = Path("/home/dipshikha/Music/cao")

# Original CISLR metadata
DATASET_CSV = DATASET_ROOT / "dataset.csv"
PROTOTYPE_CSV = DATASET_ROOT / "prototype.csv"
TEST_CSV = DATASET_ROOT / "test.csv"

# Stage 4.2 canonical manifest
STAGE4_2_MANIFEST = (
    PROJECT_ROOT
    / "audit"
    / "stage4_2_protocol"
    / "stage4_2_canonical_training_manifest.csv"
)

# Stage 4.5 official comparison
STAGE4_5_COMPARISON = (
    PROJECT_ROOT
    / "audit"
    / "stage4_5_official_evaluation"
    / "stage4_5_official_model_comparison.csv"
)

# Stage 5 output directory
STAGE5_ROOT = PROJECT_ROOT / "audit" / "stage5_hand_roi"
STAGE5_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = STAGE5_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_DIR = STAGE5_ROOT / "protocol"
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 2. EXPERIMENTAL CONSTANTS
# =============================================================================

SEED = 42

MODEL_NAME = "MobileNetV3-Large"

INPUT_WIDTH = 224
INPUT_HEIGHT = 224
INPUT_CHANNELS = 3

FRAMES_PER_VIDEO = 8

EMBEDDING_DIMENSION = 960

NUM_CLASSES = 4764

EXPECTED_PROTOTYPE_VIDEOS = 4764
EXPECTED_TEST_VIDEOS = 2285

EVALUATION_PROTOCOL = "official_prototype_to_test_retrieval"

SIMILARITY_METRIC = "cosine"

INPUT_CONDITIONS = [
    "FULL_FRAME",
    "HAND_ROI"
]

# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def sha256_file(path, chunk_size=1024 * 1024):
    """
    Calculate SHA-256 hash of a file.
    """
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def require_file(path, description):
    """
    Verify that an expected artifact exists.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{description} not found:\n{path}"
        )

    return True


# =============================================================================
# 5. VERIFY INPUT ARTIFACTS
# =============================================================================

print("=" * 80)
print("CISLR — STAGE 5.1 EXPERIMENTAL PROTOCOL")
print("=" * 80)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nVerifying required input artifacts...")

require_file(DATASET_CSV, "dataset.csv")
require_file(PROTOTYPE_CSV, "prototype.csv")
require_file(TEST_CSV, "test.csv")
require_file(STAGE4_2_MANIFEST, "Stage 4.2 canonical manifest")
require_file(STAGE4_5_COMPARISON, "Stage 4.5 official comparison")

print("dataset.csv                         : FOUND")
print("prototype.csv                      : FOUND")
print("test.csv                            : FOUND")
print("Stage 4.2 canonical manifest       : FOUND")
print("Stage 4.5 official comparison     : FOUND")

# =============================================================================
# 6. LOAD OFFICIAL DATASET FILES
# =============================================================================

print("\n" + "=" * 80)
print("LOADING CISLR METADATA")
print("=" * 80)

dataset_df = pd.read_csv(DATASET_CSV)
prototype_df = pd.read_csv(PROTOTYPE_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"dataset.csv rows    : {len(dataset_df)}")
print(f"prototype.csv rows  : {len(prototype_df)}")
print(f"test.csv rows       : {len(test_df)}")

# =============================================================================
# 7. BASIC COLUMN VALIDATION
# =============================================================================

required_dataset_columns = {"UID", "gloss"}

required_prototype_columns = {"UID", "gloss"}

required_test_columns = {"UID", "gloss"}

missing = required_dataset_columns - set(dataset_df.columns)

if missing:
    # The metadata files use lowercase `uid`, while the protocol expects `UID`.
    for df in (dataset_df, prototype_df, test_df):
        df.columns = df.columns.str.strip()

        if "UID" not in df.columns and "uid" in df.columns:
            df.rename(columns={"uid": "UID"}, inplace=True)

    missing = required_dataset_columns - set(dataset_df.columns)

    if missing:
        raise RuntimeError(
            f"dataset.csv is missing required columns: {sorted(missing)}"
        )

missing = required_prototype_columns - set(prototype_df.columns)

if missing:
    raise RuntimeError(
        f"prototype.csv is missing required columns: {sorted(missing)}"
    )

missing = required_test_columns - set(test_df.columns)

if missing:
    raise RuntimeError(
        f"test.csv is missing required columns: {sorted(missing)}"
    )

# =============================================================================
# 8. REMOVE ONLY THE KNOWN MALFORMED LABEL ROW FROM SUPERVISED METADATA
# =============================================================================
#
# IMPORTANT:
#   Raw dataset.csv is NOT modified.
#
#   The known malformed UID is:
#
#       R_GaklgD4bY=-
#
#   It has no valid gloss and therefore cannot participate in label-dependent
#   experiments.
#
# =============================================================================

MALFORMED_UID = "R_GaklgD4bY=-"

valid_dataset_df = dataset_df[
    dataset_df["UID"].notna() &
    dataset_df["gloss"].notna()
].copy()

valid_prototype_df = prototype_df[
    prototype_df["UID"].notna() &
    prototype_df["gloss"].notna()
].copy()

valid_test_df = test_df[
    test_df["UID"].notna() &
    test_df["gloss"].notna()
].copy()

print("\nValid metadata:")
print(f"Dataset valid rows   : {len(valid_dataset_df)}")
print(f"Prototype valid rows : {len(valid_prototype_df)}")
print(f"Test valid rows      : {len(valid_test_df)}")

# =============================================================================
# 9. UID UNIQUENESS CHECK
# =============================================================================

print("\n" + "=" * 80)
print("UID CONSISTENCY CHECK")
print("=" * 80)

dataset_uid_count = valid_dataset_df["UID"].nunique()
prototype_uid_count = valid_prototype_df["UID"].nunique()
test_uid_count = valid_test_df["UID"].nunique()

print(f"Dataset unique UIDs   : {dataset_uid_count}")
print(f"Prototype unique UIDs : {prototype_uid_count}")
print(f"Test unique UIDs      : {test_uid_count}")

if prototype_uid_count != EXPECTED_PROTOTYPE_VIDEOS:
    raise RuntimeError(
        f"Expected {EXPECTED_PROTOTYPE_VIDEOS} valid prototype videos, "
        f"found {prototype_uid_count}."
    )

if test_uid_count != EXPECTED_TEST_VIDEOS:
    raise RuntimeError(
        f"Expected {EXPECTED_TEST_VIDEOS} test videos, "
        f"found {test_uid_count}."
    )

# =============================================================================
# 10. PROTOTYPE / TEST OVERLAP CHECK
# =============================================================================

prototype_uids = set(valid_prototype_df["UID"])
test_uids = set(valid_test_df["UID"])

overlap = prototype_uids.intersection(test_uids)

print(f"\nPrototype/Test UID overlap : {len(overlap)}")

if len(overlap) != 0:
    raise RuntimeError(
        "Prototype and test sets overlap. "
        "This would violate the official evaluation protocol."
    )

# =============================================================================
# 11. VERIFY ONE PROTOTYPE PER CLASS
# =============================================================================

print("\n" + "=" * 80)
print("PROTOTYPE CLASS STRUCTURE")
print("=" * 80)

prototype_class_counts = (
    valid_prototype_df
    .groupby("gloss")["UID"]
    .nunique()
)

num_prototype_classes = len(prototype_class_counts)

max_prototypes_per_class = prototype_class_counts.max()
min_prototypes_per_class = prototype_class_counts.min()

print(f"Prototype classes           : {num_prototype_classes}")
print(f"Minimum videos/class        : {min_prototypes_per_class}")
print(f"Maximum videos/class        : {max_prototypes_per_class}")

if (
    num_prototype_classes != NUM_CLASSES
    or min_prototypes_per_class != 1
    or max_prototypes_per_class != 1
):
    raise RuntimeError(
        "Prototype structure does not match the expected one-shot "
        "one-prototype-per-class protocol."
    )

# =============================================================================
# 12. VERIFY TEST CLASSES
# =============================================================================

test_class_count = valid_test_df["gloss"].nunique()

print(f"Test classes                : {test_class_count}")

test_only_classes = set(valid_test_df["gloss"]) - set(valid_prototype_df["gloss"])

print(f"Test-only classes           : {len(test_only_classes)}")

if len(test_only_classes) != 0:
    raise RuntimeError(
        "Found test classes without corresponding prototype classes."
    )

# =============================================================================
# 13. LOAD STAGE 4.2 CANONICAL MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("LOADING STAGE 4.2 CANONICAL MANIFEST")
print("=" * 80)

stage4_2_df = pd.read_csv(STAGE4_2_MANIFEST)

print(f"Rows: {len(stage4_2_df)}")
print(f"Columns: {list(stage4_2_df.columns)}")

required_stage4_columns = {
    "uid",
    "gloss",
    "video_path",
    "split",
    "sample_number",
    "frame_index",
    "fps",
    "frame_count",
    "width",
    "height",
    "duration_seconds"
}

missing_stage4_columns = (
    required_stage4_columns - set(stage4_2_df.columns)
)

if missing_stage4_columns:
    raise RuntimeError(
        "Stage 4.2 manifest is missing columns:\n"
        + str(sorted(missing_stage4_columns))
    )

# =============================================================================
# 14. VERIFY STAGE 4.2 MANIFEST
# =============================================================================

manifest_video_count = stage4_2_df["uid"].nunique()
manifest_row_count = len(stage4_2_df)

expected_manifest_rows = EXPECTED_PROTOTYPE_VIDEOS * FRAMES_PER_VIDEO

print(f"\nUnique videos       : {manifest_video_count}")
print(f"Manifest rows       : {manifest_row_count}")
print(f"Expected rows       : {expected_manifest_rows}")

if manifest_video_count != EXPECTED_PROTOTYPE_VIDEOS:
    raise RuntimeError(
        "Stage 4.2 canonical manifest does not contain the expected "
        "4764-video canonical pool."
    )

if manifest_row_count != expected_manifest_rows:
    raise RuntimeError(
        "Stage 4.2 manifest does not contain exactly "
        "8 frames per video."
    )

frames_per_video = (
    stage4_2_df
    .groupby("uid")
    .size()
)

if not (frames_per_video == FRAMES_PER_VIDEO).all():
    raise RuntimeError(
        "Not every Stage 4.2 video contains exactly 8 sampled frames."
    )

# =============================================================================
# 15. VERIFY NO OFFICIAL TEST VIDEOS ARE IN TRAINING MANIFEST
# =============================================================================

training_manifest_uids = set(stage4_2_df["uid"])

training_test_overlap = training_manifest_uids.intersection(test_uids)

print(
    f"\nTraining-manifest / official-test overlap : "
    f"{len(training_test_overlap)}"
)

if len(training_test_overlap) != 0:
    raise RuntimeError(
        "Stage 4.2 training manifest overlaps with official test videos."
    )

# =============================================================================
# 16. BUILD STAGE 5 MASTER VIDEO MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING STAGE 5 MASTER MANIFEST")
print("=" * 80)

# Prototype manifest
prototype_manifest = valid_prototype_df[
    ["UID", "gloss"]
].copy()

prototype_manifest["split"] = "prototype"
prototype_manifest["input_condition"] = "BOTH"
prototype_manifest["frames_per_video"] = FRAMES_PER_VIDEO
prototype_manifest["input_width"] = INPUT_WIDTH
prototype_manifest["input_height"] = INPUT_HEIGHT
prototype_manifest["channels"] = INPUT_CHANNELS

# Test manifest
test_manifest = valid_test_df[
    ["UID", "gloss"]
].copy()

test_manifest["split"] = "test"
test_manifest["input_condition"] = "BOTH"
test_manifest["frames_per_video"] = FRAMES_PER_VIDEO
test_manifest["input_width"] = INPUT_WIDTH
test_manifest["input_height"] = INPUT_HEIGHT
test_manifest["channels"] = INPUT_CHANNELS

# Rename UID consistently
prototype_manifest = prototype_manifest.rename(
    columns={"UID": "uid"}
)

test_manifest = test_manifest.rename(
    columns={"UID": "uid"}
)

stage5_video_manifest = pd.concat(
    [prototype_manifest, test_manifest],
    ignore_index=True
)

# Add experiment metadata
stage5_video_manifest["model"] = MODEL_NAME
stage5_video_manifest["embedding_dimension"] = EMBEDDING_DIMENSION
stage5_video_manifest["evaluation_protocol"] = EVALUATION_PROTOCOL
stage5_video_manifest["similarity_metric"] = SIMILARITY_METRIC

# =============================================================================
# 17. ADD ORIGINAL VIDEO PATH
# =============================================================================

# CISLR videos are stored directly in this directory.
VIDEO_ROOT = DATASET_ROOT / "CISLR_v1.5-a_videos"

stage5_video_manifest["video_path"] = (
    stage5_video_manifest["uid"]
    .astype(str)
    .apply(lambda uid: str(VIDEO_ROOT / f"{uid}.mp4"))
)

# =============================================================================
# 18. VIDEO EXISTENCE CHECK
# =============================================================================

print("\nChecking video paths...")

stage5_video_manifest["video_exists"] = (
    stage5_video_manifest["video_path"]
    .apply(lambda p: Path(p).exists())
)

missing_video_rows = stage5_video_manifest[
    ~stage5_video_manifest["video_exists"]
]

print(
    f"Videos found : "
    f"{stage5_video_manifest['video_exists'].sum()} / "
    f"{len(stage5_video_manifest)}"
)

print(
    f"Missing videos : {len(missing_video_rows)}"
)

if len(missing_video_rows) > 0:
    print("\nWARNING: Missing video files:")
    print(missing_video_rows[["uid", "split", "video_path"]].head(20))

# Do not silently remove them.
# Missing videos will be handled explicitly by later stages.

# =============================================================================
# 19. DEFINE THE TWO STAGE 5 CONDITIONS
# =============================================================================

condition_table = pd.DataFrame([
    {
        "condition": "FULL_FRAME",
        "description": "Entire video frame is provided to MobileNetV3-Large.",
        "hand_detection": False,
        "roi_extraction": False,
        "backbone": MODEL_NAME,
        "input_size": "224x224",
        "frames_per_video": FRAMES_PER_VIDEO,
        "evaluation": EVALUATION_PROTOCOL
    },
    {
        "condition": "HAND_ROI",
        "description": "Detected hand region is cropped and provided to MobileNetV3-Large.",
        "hand_detection": True,
        "roi_extraction": True,
        "backbone": MODEL_NAME,
        "input_size": "224x224",
        "frames_per_video": FRAMES_PER_VIDEO,
        "evaluation": EVALUATION_PROTOCOL
    }
])

# =============================================================================
# 20. CREATE FROZEN EXPERIMENTAL PROTOCOL
# =============================================================================

protocol = {
    "stage": "5.1",
    "stage_name": "Hand ROI Experimental Protocol & Dataset Manifest",

    "project": "CISLR OpenVINO Research",

    "objective": (
        "Compare full-frame and hand-ROI visual representations "
        "under a controlled CISLR recognition experiment."
    ),

    "dataset": {
        "name": "CISLR v1.5-a",
        "dataset_csv": str(DATASET_CSV),
        "prototype_csv": str(PROTOTYPE_CSV),
        "test_csv": str(TEST_CSV),
        "prototype_videos": EXPECTED_PROTOTYPE_VIDEOS,
        "test_videos": EXPECTED_TEST_VIDEOS,
        "num_classes": NUM_CLASSES
    },

    "model": {
        "backbone": MODEL_NAME,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "input_shape": [
            1,
            INPUT_CHANNELS,
            INPUT_HEIGHT,
            INPUT_WIDTH
        ]
    },

    "temporal_sampling": {
        "frames_per_video": FRAMES_PER_VIDEO,
        "sampling_policy": "uniform_8_frame_sampling"
    },

    "preprocessing": {
        "resize": "224x224",
        "channels": "RGB",
        "normalization": "ImageNet",
        "evaluation_augmentation": False
    },

    "conditions": {
        "full_frame": {
            "enabled": True,
            "hand_detection": False,
            "roi_extraction": False
        },

        "hand_roi": {
            "enabled": True,
            "hand_detection": True,
            "roi_extraction": True,
            "padding_policy": "TO_BE_DEFINED_IN_STAGE_5.3"
        }
    },

    "evaluation": {
        "protocol": EVALUATION_PROTOCOL,
        "prototype_videos": EXPECTED_PROTOTYPE_VIDEOS,
        "test_videos": EXPECTED_TEST_VIDEOS,
        "similarity": SIMILARITY_METRIC,
        "metrics": [
            "Top-1",
            "Top-5",
            "Recall@10",
            "MRR",
            "mean_rank",
            "median_rank"
        ]
    },

    "controls": {
        "same_backbone": True,
        "same_frame_sampling": True,
        "same_input_resolution": True,
        "same_normalization": True,
        "same_prototype_test_split": True,
        "same_retrieval_protocol": True,
        "original_videos_modified": False
    },

    "stage5_scope": {
        "stage_5_1": "Protocol and manifest only",
        "stage_5_2": "Hand detector selection",
        "stage_5_3": "ROI extraction",
        "stage_5_4": "ROI quality validation",
        "stage_5_5": "Full-frame controlled baseline",
        "stage_5_6": "Hand-ROI model",
        "stage_5_7": "Full-frame vs ROI comparison",
        "stage_5_8": "Final analysis and selection"
    },

    "reproducibility": {
        "seed": SEED,
        "python_version": sys.version,
        "platform": platform.platform(),
        "architecture": platform.machine(),
        "created_at": datetime.now().isoformat()
    },

    "scientific_notes": [
        "Stage 5 does not replace or modify the official CISLR test split.",
        "Stage 4 results remain frozen and are not rerun.",
        "The full-frame MobileNetV3-Large result from Stage 4.5 is retained as the existing reference.",
        "Hand ROI performance will be evaluated as a new experimental condition.",
        "Hand detection and ROI extraction costs must be measured separately in later stages.",
        "A failed hand detection must not be silently treated as a successful ROI.",
        "Original CISLR videos remain untouched."
    ]
}

# =============================================================================
# 21. SAVE MANIFESTS
# =============================================================================

MASTER_MANIFEST_PATH = (
    MANIFEST_DIR / "stage5_1_master_video_manifest.csv"
)

CONDITION_PATH = (
    PROTOCOL_DIR / "stage5_1_experimental_conditions.csv"
)

PROTOCOL_PATH = (
    PROTOCOL_DIR / "stage5_1_protocol.json"
)

stage5_video_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False
)

condition_table.to_csv(
    CONDITION_PATH,
    index=False
)

with open(PROTOCOL_PATH, "w") as f:
    json.dump(protocol, f, indent=2)

# =============================================================================
# 22. SAVE INPUT ARTIFACT HASHES
# =============================================================================

hash_records = {
    "dataset_csv_sha256": sha256_file(DATASET_CSV),
    "prototype_csv_sha256": sha256_file(PROTOTYPE_CSV),
    "test_csv_sha256": sha256_file(TEST_CSV),
    "stage4_2_manifest_sha256": sha256_file(STAGE4_2_MANIFEST),
    "stage4_5_comparison_sha256": sha256_file(STAGE4_5_COMPARISON)
}

HASH_PATH = (
    PROTOCOL_DIR / "stage5_1_input_artifact_hashes.json"
)

with open(HASH_PATH, "w") as f:
    json.dump(hash_records, f, indent=2)

# =============================================================================
# 23. SAVE MANIFEST SUMMARY
# =============================================================================

summary = {
    "stage": "5.1",
    "status": "COMPLETE",

    "dataset_rows": int(len(dataset_df)),
    "valid_dataset_rows": int(len(valid_dataset_df)),

    "prototype_videos": int(prototype_uid_count),
    "test_videos": int(test_uid_count),

    "prototype_test_overlap": int(len(overlap)),

    "prototype_classes": int(num_prototype_classes),
    "test_classes": int(test_class_count),

    "stage4_2_training_manifest_videos": int(
        manifest_video_count
    ),

    "stage4_2_training_manifest_frames": int(
        manifest_row_count
    ),

    "frames_per_video": FRAMES_PER_VIDEO,

    "model": MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIMENSION,

    "conditions": [
        "FULL_FRAME",
        "HAND_ROI"
    ],

    "input_resolution": "224x224",

    "evaluation_protocol": EVALUATION_PROTOCOL,

    "master_manifest": str(MASTER_MANIFEST_PATH),
    "conditions_file": str(CONDITION_PATH),
    "protocol_file": str(PROTOCOL_PATH),
    "hash_file": str(HASH_PATH)
}

SUMMARY_PATH = (
    PROTOCOL_DIR / "stage5_1_summary.json"
)

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

# =============================================================================
# 24. FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("STAGE 5.1 VALIDATION")
print("=" * 80)

checks = {
    "4764 prototype videos": prototype_uid_count == 4764,
    "2285 test videos": test_uid_count == 2285,
    "0 prototype/test overlap": len(overlap) == 0,
    "4764 prototype classes": num_prototype_classes == 4764,
    "1 prototype/class": max_prototypes_per_class == 1,
    "0 test-only classes": len(test_only_classes) == 0,
    "Stage 4.2 has 4764 videos": manifest_video_count == 4764,
    "Stage 4.2 has 38112 frames": manifest_row_count == 38112,
    "8 frames/video": (frames_per_video == 8).all(),
    "MobileNet embedding = 960": EMBEDDING_DIMENSION == 960,
    "Input = 224x224": (
        INPUT_WIDTH == 224 and INPUT_HEIGHT == 224
    )
}

all_pass = True

for name, passed in checks.items():
    print(f"{name:<45} : {'PASS' if passed else 'FAIL'}")

    if not passed:
        all_pass = False

print("\n" + "=" * 80)

if not all_pass:
    raise RuntimeError(
        "Stage 5.1 validation failed. "
        "Do not proceed to Stage 5.2."
    )

print("STAGE 5.1 COMPLETE")
print("=" * 80)

print("\nFROZEN EXPERIMENT:")
print("  FULL_FRAME  vs  HAND_ROI")

print("\nMODEL:")
print(f"  {MODEL_NAME}")

print("\nINPUT:")
print(f"  {FRAMES_PER_VIDEO} frames/video")
print(f"  {INPUT_WIDTH}x{INPUT_HEIGHT} RGB")

print("\nOFFICIAL EVALUATION:")
print(f"  Prototype videos : {EXPECTED_PROTOTYPE_VIDEOS}")
print(f"  Test videos      : {EXPECTED_TEST_VIDEOS}")
print(f"  Classes          : {NUM_CLASSES}")

print("\nOUTPUTS:")
print(f"  Master manifest  : {MASTER_MANIFEST_PATH}")
print(f"  Conditions       : {CONDITION_PATH}")
print(f"  Protocol         : {PROTOCOL_PATH}")
print(f"  Hashes           : {HASH_PATH}")
print(f"  Summary          : {SUMMARY_PATH}")

print("\nNEXT:")
print("  >>> STAGE 5.2 — HAND DETECTOR SELECTION")
print("=" * 80)

CISLR — STAGE 5.1 EXPERIMENTAL PROTOCOL

Project root:
/home/dipshikha/Music/cao/CISLR_RESEARCH

Verifying required input artifacts...
dataset.csv                         : FOUND
prototype.csv                      : FOUND
test.csv                            : FOUND
Stage 4.2 canonical manifest       : FOUND
Stage 4.5 official comparison     : FOUND

LOADING CISLR METADATA
dataset.csv rows    : 7050
prototype.csv rows  : 4765
test.csv rows       : 2285

Valid metadata:
Dataset valid rows   : 7049
Prototype valid rows : 4764
Test valid rows      : 2285

UID CONSISTENCY CHECK
Dataset unique UIDs   : 7049
Prototype unique UIDs : 4764
Test unique UIDs      : 2285

Prototype/Test UID overlap : 0

PROTOTYPE CLASS STRUCTURE
Prototype classes           : 4764
Minimum videos/class        : 1
Maximum videos/class        : 1
Test classes                : 1363
Test-only classes           : 0

LOADING STAGE 4.2 CANONICAL MANIFEST
Rows: 38112
Columns: ['uid', 'gloss', 'video_path', 'split', 'sample_n